# Trading Lab — Data Exploration Notebook

## Purpose

This notebook is the primary research and exploration tool for the Trading Lab system. It allows you to:

- Download and inspect historical price data for commodity instruments
- Visualise price charts and technical indicators
- Generate and review trading signals produced by the SMA crossover strategy
- Run backtests and evaluate strategy performance with a full suite of metrics
- Validate whether a strategy generalises to new data using in-sample / out-of-sample testing

## How to use this notebook

Run cells from top to bottom. The key variables you will want to change are defined at the top of each section:

- **`SYMBOL`** — which commodity to analyse
- **`PERIOD`** — how much historical data to download
- **`FAST` / `SLOW`** — the SMA window parameters for the strategy

## Important rules

| Rule | Reason |
|------|--------|
| This notebook is for **exploration only** | Production logic belongs in `src/trading_lab/` |
| **Clear outputs before committing** | Parquet data embedded in outputs bloats the repo |
| Do not define strategy classes or data pipelines here | Notebooks are not version-controlled reliably |

To clear outputs before committing: `Kernel → Restart & Clear Output`

---
## Setup

This cell imports all required modules from the `trading_lab` package and configures matplotlib.

**If this cell fails with `ModuleNotFoundError`:**
1. Make sure your virtual environment is activated: `.venv\Scripts\Activate.ps1`
2. Install dependencies: `pip install -e .`
3. Restart the kernel and try again

The `ensure_data_dirs()` call creates all required data directories (`data/raw/`, `data/curated/`, etc.) if they do not already exist.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import matplotlib.pyplot as plt

from trading_lab.data.models import MarketDataRequest
from trading_lab.data.yfinance_ingest import ingest_yfinance_daily, fetch_news
from trading_lab.features.indicators import sma, rsi, atr
from trading_lab.strategies.sma_cross import SmaCrossStrategy
from trading_lab.backtesting.engine import run_backtest
from trading_lab.backtesting.models import BacktestConfig
from trading_lab.backtesting.metrics import compute_all
from trading_lab.backtesting.validation import (
    split_in_sample_out_of_sample,
    validate_oos_thresholds,
    compute_performance_degradation,
)
from trading_lab.llm.stub_client import StubLLMClient
from trading_lab.llm.claude_client import ClaudeClient
from trading_lab.llm.context import build_signal_context
from trading_lab.llm.explainer import ExplanationService
from trading_lab.llm.decision import DecisionService
from trading_lab.paths import ensure_data_dirs, EXPLANATIONS_DIR, DECISIONS_DIR

ensure_data_dirs()

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Setup complete')

---
## Section 1 — Download Market Data

This section downloads historical daily price data from **Yahoo Finance** via the `yfinance` library and saves it locally as Parquet files.

### What happens behind the scenes

1. A `MarketDataRequest` is created with your chosen symbol, period, and settings
2. `ingest_yfinance_daily()` downloads the data and saves two files:
   - `data/raw/<symbol>_1d_yfinance.parquet` — the raw data exactly as received from Yahoo Finance
   - `data/curated/<symbol>_1d_yfinance.parquet` — normalised to the project schema (UTC timestamps, standard column names, adjusted flag)
3. The curated DataFrame is returned for use in this notebook

### Available instruments (Phase 1)

| Symbol | Instrument | Notes |
|--------|------------|-------|
| `GC=F` | Gold | Most liquid, strong trending behaviour |
| `CL=F` | Crude Oil (WTI) | Highly volatile, sensitive to geopolitical events |
| `SI=F` | Silver | Correlated with Gold but more volatile |
| `HG=F` | Copper | Industrial commodity, tracks global growth |
| `NG=F` | Natural Gas | Extremely volatile, seasonal patterns |

### Period options

| Period | Description | Notes |
|--------|-------------|-------|
| `'6mo'` | 6 months | Minimum recommended for SMA 20/50 strategy |
| `'1y'` | 1 year | Good for initial exploration |
| `'2y'` | 2 years | Recommended — covers different market conditions |
| `'5y'` | 5 years | Better for strategy validation |
| `'max'` | All available | Use for thorough backtesting |

> **Note:** Commodity futures data (`=F` symbols) uses unadjusted prices (`adjusted=False`). This is correct — commodities do not have dividend adjustments like equities do.

In [ ]:
SYMBOL = 'GC=F'   # Change to explore other instruments
PERIOD = '2y'     # How much historical data to download
FAST   = 20       # Fast SMA window (days)
SLOW   = 50       # Slow SMA window (days)

request = MarketDataRequest(symbol=SYMBOL, period=PERIOD, interval='1d', adjusted=False)
_, _, df = ingest_yfinance_daily(request)

print(f'Symbol:  {SYMBOL}')
print(f'Bars:    {len(df)} trading days')
print(f'From:    {df.index.min().date()}')
print(f'To:      {df.index.max().date()}')
print(f'Columns: {list(df.columns)}')
print()
df.tail()

---
## Section 2 — Price Chart

A simple chart of the daily closing price over the selected period.

### What to look for

- **Trend direction** — is the instrument broadly trending up, down, or sideways?
- **Trend clarity** — a clear, sustained trend is where SMA crossover strategies perform best. Choppy, sideways markets produce false signals and losses.
- **Volatility** — how large are the daily price swings? This affects stop loss sizing.
- **Major events** — sharp vertical moves often correspond to news events (Fed decisions, geopolitical events, supply reports). The strategy has no awareness of these.

> **Tip:** Gold (`GC=F`) tends to show cleaner trends than Natural Gas (`NG=F`), which is highly seasonal and erratic. Start with Gold when learning the system.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df['close'], linewidth=1.2, color='steelblue')
ax.set_title(f'{SYMBOL} — Daily Close Price ({PERIOD})')
ax.set_ylabel('Price')
plt.tight_layout()
plt.show()

---
## Section 3 — SMA Indicator Overlay

This chart overlays two Simple Moving Averages (SMAs) on the price series — the same indicators used by the trading strategy.

### What is an SMA?

A Simple Moving Average smooths out daily price noise by averaging the closing price over a set number of days. For example, the 20-day SMA is the average of the last 20 closing prices.

- **Fast SMA (20-day)** — reacts quickly to recent price changes. Shown in orange.
- **Slow SMA (50-day)** — reflects the longer-term trend. Shown in green.

### How signals are generated from the crossover

| Condition | Signal | Meaning |
|-----------|--------|---------|
| Fast SMA crosses **above** Slow SMA | **LONG** | Short-term momentum is above the long-term trend — bullish |
| Fast SMA crosses **below** Slow SMA | **SHORT** | Short-term momentum is below the long-term trend — bearish |

### What to look for

- **Crossover points** — where the orange line crosses the green line. These are your signal triggers.
- **Separation** — the wider the gap between the two SMAs, the stronger the trend and the higher the signal confidence score.
- **Whipsaws** — periods where the lines cross back and forth rapidly in a sideways market. These generate costly false signals.

### Tuning

Change `FAST` and `SLOW` in Section 1 to experiment:
- **Smaller windows** (e.g. 10/20) — more signals, react faster, more noise
- **Larger windows** (e.g. 50/200) — fewer signals, slower to react, better in strong trends

In [ ]:
fast_sma = sma(df['close'], FAST)
slow_sma = sma(df['close'], SLOW)

fig, ax = plt.subplots()
ax.plot(df.index, df['close'], label='Close', linewidth=1, alpha=0.6, color='steelblue')
ax.plot(df.index, fast_sma, label=f'SMA {FAST} (fast)', linewidth=1.5, color='orange')
ax.plot(df.index, slow_sma, label=f'SMA {SLOW} (slow)', linewidth=1.5, color='green')
ax.set_title(f'{SYMBOL} — Price with SMA {FAST}/{SLOW} Crossover')
ax.set_ylabel('Price')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 4 — RSI (Relative Strength Index)

The RSI is a momentum indicator that measures the speed and magnitude of recent price changes. It is used in this system as a **filter** — it does not generate signals on its own, but it can suppress signals from the SMA crossover when conditions are unfavourable.

### How RSI is calculated

RSI compares the average size of recent gains to recent losses over a 14-day period and produces a value between 0 and 100:

- **RSI > 70** — the instrument is considered **overbought**. The price has risen rapidly and may be due for a pullback. A long signal generated here has a higher chance of being a false entry.
- **RSI < 30** — the instrument is considered **oversold**. The price has fallen rapidly. A short signal generated here may be late.
- **RSI 40–60** — neutral zone. Signals in this range are considered higher quality.

### How the RSI filter works in this strategy

| SMA signal | RSI condition | Result |
|------------|--------------|--------|
| LONG | RSI ≤ 70 | Signal passes through |
| LONG | RSI > 70 | **Signal suppressed** — overbought |
| SHORT | RSI ≥ 30 | Signal passes through |
| SHORT | RSI < 30 | **Signal suppressed** — oversold |

### What to look for

- Periods where RSI spends time above 70 or below 30 — these are where the filter is actively working
- Compare signal counts in Section 6 with and without the RSI filter to see its effect
- A `conflicting_indicators` warning in the signal output means the SMA and RSI are pointing in opposite directions

In [ ]:
rsi_values = rsi(df['close'], window=14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(df.index, df['close'], linewidth=1, color='steelblue')
ax1.set_title(f'{SYMBOL} — Price')
ax1.set_ylabel('Price')

ax2.plot(df.index, rsi_values, color='purple', linewidth=1)
ax2.axhline(70, color='red',   linestyle='--', alpha=0.7, label='Overbought (70) — long signals suppressed above here')
ax2.axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30) — short signals suppressed below here')
ax2.axhline(50, color='grey',  linestyle=':',  alpha=0.5, label='Midpoint (50)')
ax2.fill_between(df.index, 70, 100, alpha=0.05, color='red')
ax2.fill_between(df.index, 0, 30,   alpha=0.05, color='green')
ax2.set_title('RSI (14) — Momentum Filter')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Current RSI: {rsi_values.iloc[-1]:.1f}')

---
## Section 5 — ATR (Average True Range — Volatility)

The ATR measures market volatility — specifically, how much the price typically moves each day. It is used in this system for two purposes:

1. **Stop loss sizing** — the stop loss is placed at 1.5× ATR away from the entry price, ensuring it is wide enough not to be hit by normal daily price noise
2. **High volatility detection** — if the current ATR exceeds 2× its 30-day average, the signal is flagged as HIGH VOLATILITY

### How ATR is calculated

For each bar, the True Range is the largest of:
- Current high minus current low
- Absolute value of current high minus previous close
- Absolute value of current low minus previous close

The ATR is the 14-day average of these True Range values.

### What to look for

- **Rising ATR** — volatility is increasing. Stop losses will be wider, position sizes smaller.
- **Spikes** — sudden ATR spikes often correspond to major news events. Signals during spikes are flagged as HIGH VOLATILITY and should be treated with extra caution.
- **Current vs average** — if current ATR is much higher than the 30-day average printed below the chart, be aware that stops will be unusually wide.

### Effect on stop loss

A higher ATR = wider stop loss = larger potential loss per trade. The position sizing calculator accounts for this by reducing your position size when ATR is high, keeping your £ risk per trade constant.

In [ ]:
atr_values = atr(df['high'], df['low'], df['close'], window=14)
atr_avg_30 = atr_values.rolling(30).mean()

fig, ax = plt.subplots()
ax.plot(df.index, atr_values, color='orange', linewidth=1, label='ATR (14)')
ax.plot(df.index, atr_avg_30, color='red', linewidth=1.2, linestyle='--', label='30-day ATR average')
ax.fill_between(df.index, atr_avg_30 * 2, atr_values,
                where=atr_values > atr_avg_30 * 2,
                alpha=0.2, color='red', label='High volatility zone (ATR > 2x average)')
ax.set_title(f'{SYMBOL} — ATR (14) Volatility Indicator')
ax.set_ylabel('ATR Value')
ax.legend()
plt.tight_layout()
plt.show()

current_atr = atr_values.iloc[-1]
avg_atr = atr_avg_30.iloc[-1]
print(f'Current ATR:      {current_atr:.2f}')
print(f'30-day avg ATR:   {avg_atr:.2f}')
print(f'High volatility:  {"YES — ATR > 2x average" if current_atr > avg_atr * 2 else "No"}')
print(f'Stop distance:    {current_atr * 1.5:.2f} (1.5x ATR)')

---
## Section 6 — Signal Generation

This section runs the full `SmaCrossStrategy` against the price data and displays the results.

### What the strategy produces

For every bar in the dataset, the strategy calculates:

| Field | Description |
|-------|-------------|
| `signal` | Trading direction: `1` = Long, `-1` = Short, `0` = Flat |
| `rsi` | RSI value used by the filter |
| `confidence_score` | 0–100 score based on how strongly indicators agree |
| `signal_strength_pct` | SMA gap as % of price — wider gap = stronger trend |
| `conflicting_indicators` | True if SMA says one direction but RSI suggests caution |
| `high_volatility` | True if ATR is more than 2x its 30-day average |
| `stop_loss_level` | Price level to exit the trade if it goes against you |
| `take_profit_level` | Price level to exit the trade with your target profit |

### Confidence score breakdown

| Component | Points | Condition |
|-----------|--------|-----------|
| Base score | 50 | All non-flat signals start here |
| SMA gap bonus | +25 | SMA gap > 1% of price (strong separation) |
| RSI direction bonus | +25 | RSI confirms signal direction (RSI < 40 for long, RSI > 60 for short) |
| **Maximum** | **100** | All three conditions met |

### Reading the signal chart

- **Green triangles (▲)** — long signal fired (buy)
- **Red triangles (▼)** — short signal fired (sell)
- **Grey circles (●)** — signal changed to flat (exit)

Note that signals are **persistent** — once a long signal fires, the position remains long until a new signal fires. The markers show where the signal *changed*, not every bar.

In [ ]:
strategy = SmaCrossStrategy(fast_window=FAST, slow_window=SLOW)
signals = strategy.generate_signals(df)

counts = signals['signal'].value_counts().sort_index()
total = len(signals)
print(f'Signal breakdown over {total} bars:')
print(f'  Long  (+1): {counts.get(1, 0):>4} bars ({counts.get(1, 0)/total*100:.1f}%)')
print(f'  Flat   (0): {counts.get(0, 0):>4} bars ({counts.get(0, 0)/total*100:.1f}%)')
print(f'  Short (-1): {counts.get(-1, 0):>4} bars ({counts.get(-1, 0)/total*100:.1f}%)')

In [ ]:
# Most recent 5 bars with full signal detail
latest = signals[[
    'close', 'signal', 'rsi',
    'confidence_score', 'signal_strength_pct',
    'conflicting_indicators', 'high_volatility',
    'stop_loss_level', 'take_profit_level'
]].tail(5)
latest

In [ ]:
# Plain-English summary of the current (latest) signal
last = signals.iloc[-1]
sig_map = {1: 'LONG  ▲', -1: 'SHORT ▼', 0: 'FLAT  —'}

print(f"{'='*40}")
print(f"  {SYMBOL} — Current Signal Summary")
print(f"{'='*40}")
print(f"  Signal:            {sig_map.get(int(last['signal']), 'FLAT')}")
print(f"  Date:              {signals.index[-1].date()}")
print(f"  Close price:       {last['close']:.2f}")
print(f"  RSI (14):          {last['rsi']:.1f}")
print(f"  Confidence:        {last['confidence_score']}/100")
print(f"  Signal strength:   {last['signal_strength_pct']:.2f}% SMA gap")
print(f"  Conflicting:       {'⚠ Yes' if last['conflicting_indicators'] else 'No'}")
print(f"  High volatility:   {'⚠ Yes' if last['high_volatility'] else 'No'}")
if int(last['signal']) != 0:
    print(f"  Stop loss:         {last['stop_loss_level']:.2f}")
    print(f"  Take profit:       {last['take_profit_level']:.2f}")
    rr = abs(last['take_profit_level'] - last['close']) / abs(last['stop_loss_level'] - last['close'])
    print(f"  Risk/Reward:       1:{rr:.1f}")
print(f"{'='*40}")

In [ ]:
# Price chart with signal entry/exit markers
changes = signals[signals['signal'].diff() != 0]
longs  = changes[changes['signal'] == 1]
shorts = changes[changes['signal'] == -1]
exits  = changes[changes['signal'] == 0]

fig, ax = plt.subplots()
ax.plot(df.index, df['close'], linewidth=1, alpha=0.7, label='Close', color='steelblue')
ax.plot(df.index, fast_sma, linewidth=1, label=f'SMA {FAST}', color='orange')
ax.plot(df.index, slow_sma, linewidth=1, label=f'SMA {SLOW}', color='green')
ax.scatter(longs.index,  longs['close'],  marker='^', color='green', s=100, zorder=5, label='Long entry')
ax.scatter(shorts.index, shorts['close'], marker='v', color='red',   s=100, zorder=5, label='Short entry')
ax.scatter(exits.index,  exits['close'],  marker='o', color='grey',  s=50,  zorder=5, label='Exit')
ax.set_title(f'{SYMBOL} — SMA {FAST}/{SLOW} Signals with RSI Filter')
ax.set_ylabel('Price')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 7 — Backtest & Performance Metrics

This section simulates how the strategy would have performed historically, applying realistic trading costs.

### How the backtest engine works

The backtest engine replays history bar by bar:

1. A signal is generated on bar N (end of day)
2. The position change takes effect at bar N+1 (next trading day)
3. This **one-bar lag** prevents lookahead bias — you cannot trade on information you don't yet have
4. On every position change, transaction costs are applied

### Transaction costs

| Cost | Default | What it represents |
|------|---------|--------------------|
| Commission | 5 bps (0.05%) | IG spreadbetting spread/commission per trade |
| Slippage | 2 bps (0.02%) | Price difference between signal price and execution price |
| **Total** | **7 bps (0.07%)** | Applied each time the position changes |

### Performance metrics explained

| Metric | What it means | Good value |
|--------|--------------|------------|
| **Total return %** | Overall profit/loss over the period | Depends on timeframe |
| **CAGR %** | Compound annual growth rate — annualised return | > 10% is reasonable |
| **Sharpe ratio** | Return per unit of risk. Higher = better risk-adjusted performance | > 1.0 is good, > 2.0 is excellent |
| **Max drawdown %** | Largest peak-to-trough loss. Lower = better | < 20% is desirable |
| **Win rate %** | Percentage of trading days that were profitable | > 50% is positive |
| **Profit factor** | Ratio of total gains to total losses | > 1.5 is good |

### Important caveats

- **Past performance does not predict future results.** A strong backtest on 2 years of data may not hold going forward.
- **Overfitting risk:** If you keep changing parameters until the backtest looks good, you are fitting to historical noise, not finding a genuine edge. Use the out-of-sample validation in Section 8 to guard against this.
- **Gold has trended strongly upward** over the past 2 years. A trend-following strategy will naturally look good in this environment. Test on other instruments and longer periods.

In [ ]:
config = BacktestConfig(
    symbol=SYMBOL,
    strategy_name='sma_cross',
    timeframe='1d',
    initial_cash=100_000,
    commission_bps=5,
    slippage_bps=2,
    strategy_params={'fast_window': FAST, 'slow_window': SLOW},
)

result  = run_backtest(signals_df=signals, config=config)
metrics = compute_all(result)

print(f"{'='*45}")
print(f"  Backtest Results — {SYMBOL} ({PERIOD})")
print(f"  Strategy: SMA {FAST}/{SLOW} with RSI filter")
print(f"{'='*45}")
print(f"  Initial capital:   £{metrics['initial_cash']:>10,.2f}")
print(f"  Final equity:      £{metrics['final_equity']:>10,.2f}")
print(f"{'─'*45}")
print(f"  Total return:       {metrics['total_return_pct']:>9.1f}%")
print(f"  CAGR:               {metrics['cagr_pct']:>9.1f}%")
print(f"  Sharpe ratio:       {metrics['sharpe_ratio']:>9.2f}")
print(f"  Max drawdown:       {metrics['max_drawdown_pct']:>9.1f}%")
print(f"  Win rate:           {metrics['win_rate_pct']:>9.1f}%")
print(f"  Profit factor:      {metrics['profit_factor']:>9.2f}")
print(f"{'='*45}")

In [ ]:
# Equity curve — shows how portfolio value grew (or fell) over time
equity = result.signals_df['equity'].dropna()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Equity curve
ax1.plot(equity.index, equity, color='steelblue', linewidth=1.5, label='Portfolio value')
ax1.axhline(config.initial_cash, color='grey', linestyle='--', alpha=0.5, label='Starting capital')
ax1.set_title(f'{SYMBOL} — Equity Curve')
ax1.set_ylabel('Portfolio Value (£)')
ax1.legend()

# Drawdown
peak = equity.cummax()
drawdown = (equity - peak) / peak * 100
ax2.fill_between(drawdown.index, drawdown, 0, alpha=0.4, color='red')
ax2.plot(drawdown.index, drawdown, color='red', linewidth=0.8)
ax2.set_title('Drawdown %')
ax2.set_ylabel('Drawdown (%)')

plt.tight_layout()
plt.show()

---
## Section 8 — In-Sample / Out-of-Sample Validation

This is the most important section for determining whether a strategy is genuinely profitable or just overfitted to historical data.

### The overfitting problem

When developing a strategy, it is tempting to keep adjusting the parameters (e.g. changing `FAST` from 20 to 15, `SLOW` from 50 to 45) until the backtest looks good. The danger is that you are not finding a genuine trading edge — you are **fitting to historical noise**. A strategy tuned this way will often fail on new, unseen data.

### The solution: hold out data the strategy never sees

The data is split chronologically into two periods:

| Period | Data | Purpose |
|--------|------|---------|
| **In-sample (70%)** | Earlier historical data | Use this to develop and tune parameters |
| **Out-of-sample (30%)** | Most recent data | Use this to test whether the strategy works on new data |

The critical rule: **you must finalise your parameters on the in-sample data before looking at the out-of-sample results.** If you adjust parameters after seeing the out-of-sample results, the test is invalid.

### Approval thresholds

A strategy must meet both criteria on the **out-of-sample period** before paper trading is permitted:

| Threshold | Minimum | Reason |
|-----------|---------|--------|
| Sharpe ratio | > 0.5 | Confirms positive risk-adjusted return |
| Max drawdown | < 25% | Limits downside risk |

### Overfitting warning

If the out-of-sample Sharpe ratio is more than 50% lower than the in-sample Sharpe ratio, an **overfitting warning** is triggered. This means the strategy is likely fitting to historical patterns rather than a genuine, repeatable edge. In this case, do not proceed to paper trading — reconsider the strategy design.

In [ ]:
# Split data 70% in-sample, 30% out-of-sample (chronological — most recent data held out)
in_sample_df, oos_df = split_in_sample_out_of_sample(df, oos_ratio=0.3)

print(f'In-sample:      {in_sample_df.index.min().date()} → {in_sample_df.index.max().date()} ({len(in_sample_df)} bars)')
print(f'Out-of-sample:  {oos_df.index.min().date()} → {oos_df.index.max().date()} ({len(oos_df)} bars)')

# Run strategy on each period
is_signals  = strategy.generate_signals(in_sample_df)
oos_signals = strategy.generate_signals(oos_df)

is_config = BacktestConfig(
    symbol=SYMBOL, strategy_name='sma_cross', timeframe='1d',
    initial_cash=100_000, commission_bps=5, slippage_bps=2,
    mode='in_sample', strategy_params={'fast_window': FAST, 'slow_window': SLOW}
)
oos_config = BacktestConfig(
    symbol=SYMBOL, strategy_name='sma_cross', timeframe='1d',
    initial_cash=100_000, commission_bps=5, slippage_bps=2,
    mode='out_of_sample', strategy_params={'fast_window': FAST, 'slow_window': SLOW}
)

is_result   = run_backtest(signals_df=is_signals,  config=is_config)
oos_result  = run_backtest(signals_df=oos_signals, config=oos_config)
is_metrics  = compute_all(is_result)
oos_metrics = compute_all(oos_result)

print()
print(f"{'Metric':<25} {'In-Sample':>12} {'Out-of-Sample':>14} {'Change':>8}")
print('─' * 62)

def _chg(a, b): return f"{b - a:+.1f}"
def _chgf(a, b): return f"{b - a:+.2f}"

print(f"{'Total return %':<25} {is_metrics['total_return_pct']:>11.1f}% {oos_metrics['total_return_pct']:>13.1f}% {_chg(is_metrics['total_return_pct'], oos_metrics['total_return_pct']):>7}%")
print(f"{'CAGR %':<25} {is_metrics['cagr_pct']:>11.1f}% {oos_metrics['cagr_pct']:>13.1f}% {_chg(is_metrics['cagr_pct'], oos_metrics['cagr_pct']):>7}%")
print(f"{'Sharpe ratio':<25} {is_metrics['sharpe_ratio']:>12.2f} {oos_metrics['sharpe_ratio']:>14.2f} {_chgf(is_metrics['sharpe_ratio'], oos_metrics['sharpe_ratio']):>8}")
print(f"{'Max drawdown %':<25} {is_metrics['max_drawdown_pct']:>11.1f}% {oos_metrics['max_drawdown_pct']:>13.1f}% {_chg(is_metrics['max_drawdown_pct'], oos_metrics['max_drawdown_pct']):>7}%")
print(f"{'Win rate %':<25} {is_metrics['win_rate_pct']:>11.1f}% {oos_metrics['win_rate_pct']:>13.1f}% {_chg(is_metrics['win_rate_pct'], oos_metrics['win_rate_pct']):>7}%")

In [ ]:
# Out-of-sample approval check
validation = validate_oos_thresholds(oos_result)

print(f"{'='*45}")
print(f"  OOS Validation Result")
print(f"{'='*45}")
status = 'APPROVED ✓ — Ready for paper trading' if validation.approved else 'NOT APPROVED ✗ — Do not proceed to paper trading'
print(f"  Status:        {status}")
print(f"  Sharpe ratio:  {validation.sharpe_ratio:.2f} (minimum: 0.5)")
print(f"  Max drawdown:  {validation.max_drawdown_pct:.1f}% (maximum: 25%)")
print(f"  Reason:        {validation.reason}")

# Overfitting check using the proper function
degradation = compute_performance_degradation(is_result, oos_result)
if degradation is not None:
    print(f"  Degradation:   {degradation:.1f}%", end='')
    if degradation > 50:
        print(' ⚠ OVERFITTING WARNING — performance dropped >50% out-of-sample')
    else:
        print(' (within acceptable range)')
else:
    print('  Degradation:   N/A (in-sample Sharpe ≤ 0)')
print(f"{'='*45}")

In [ ]:
# Equity curves: in-sample vs out-of-sample
is_equity  = is_result.signals_df['equity'].dropna()
oos_equity = oos_result.signals_df['equity'].dropna()

fig, ax = plt.subplots()
ax.plot(is_equity.index,  is_equity,  color='steelblue', linewidth=1.5, label='In-sample')
ax.plot(oos_equity.index, oos_equity, color='orange',    linewidth=1.5, label='Out-of-sample', linestyle='--')
ax.axhline(100_000, color='grey', linestyle=':', alpha=0.5, label='Starting capital')
ax.axvline(oos_equity.index[0], color='black', linestyle='--', alpha=0.3, label='OOS split point')
ax.set_title(f'{SYMBOL} — In-Sample vs Out-of-Sample Equity Curves')
ax.set_ylabel('Portfolio Value (£)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 9 — Recent News Headlines

The `fetch_news()` function retrieves the latest news headlines for an instrument via yfinance. These headlines are passed to the LLM layer to enrich signal explanations with market context.

### What you will see

- Up to 5 recent headlines with title, source, and publish timestamp
- If no headlines are available (e.g. for less-covered instruments), an empty list is returned — this is expected and handled gracefully

### How news flows through the system

In production (`scripts/run_signals.py`), news is fetched once per instrument per run and stored in the portfolio snapshot. The LLM layer reads from that snapshot — it does not make a second network call.

In [ ]:
headlines = fetch_news(SYMBOL, max_headlines=5)

print(f"Recent news for {SYMBOL} ({len(headlines)} headline(s) found):")
print()
if headlines:
    for i, h in enumerate(headlines, 1):
        import datetime
        ts = h.get('timestamp', '')
        if isinstance(ts, (int, float)) and ts:
            ts = datetime.datetime.fromtimestamp(ts, tz=datetime.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
        print(f"  {i}. {h.get('title', '—')}")
        print(f"     Source: {h.get('source', '—')}  |  Published: {ts}")
        print()
else:
    print("  No headlines available for this instrument.")

---
## Section 10 — LLM Signal Explanation

The LLM layer generates a plain-English explanation of the most recent signal. It combines the technical indicator readings with any available news headlines into a 3–5 sentence narrative.

### Two client options

| Client | When to use | Requirements |
|--------|------------|--------------|
| `StubLLMClient` | Testing, offline, no API key | None — returns a fixed placeholder string |
| `ClaudeClient` | Real explanations | `ANTHROPIC_API_KEY` environment variable must be set |

### Cache behaviour

Explanations are cached to `data/signals/explanations/<symbol>_<date>.json`. If a cached file exists for today's date, it is returned immediately without calling the API. This means you can run this cell multiple times without incurring API costs.

### What the explanation covers

The prompt instructs the model to write in two parts:
1. **Technical basis** — why the signal fired based on SMA crossover, RSI position, and signal strength
2. **News context** — whether the recent headlines support, contradict, or are neutral relative to the signal direction. Contradictions are flagged explicitly.

In [ ]:
import os
from datetime import date

# --- Choose your LLM client ---
# Use StubLLMClient if you don't have an API key, or ClaudeClient for real explanations.
if os.environ.get('ANTHROPIC_API_KEY'):
    llm_client = ClaudeClient()
    print('Using ClaudeClient (real API)')
else:
    llm_client = StubLLMClient()
    print('ANTHROPIC_API_KEY not set — using StubLLMClient (placeholder response)')

# --- Build the signal context from the most recent signal bar ---
last_row = signals.iloc[-1]
instrument_config = {'symbol': SYMBOL, 'name': SYMBOL}  # minimal; full config is in instruments.yaml

context = build_signal_context(
    signal_row={
        'signal': int(last_row['signal']),
        'close': float(last_row['close']),
        'fast_sma': float(last_row['fast_sma']),
        'slow_sma': float(last_row['slow_sma']),
        'rsi': float(last_row['rsi']) if not pd.isna(last_row['rsi']) else 50.0,
        'stop_loss_level': float(last_row['stop_loss_level']) if not pd.isna(last_row.get('stop_loss_level', float('nan'))) else 0.0,
        'take_profit_level': float(last_row['take_profit_level']) if not pd.isna(last_row.get('take_profit_level', float('nan'))) else 0.0,
        'confidence_score': int(last_row.get('confidence_score', 0)),
        'conflicting_indicators': bool(last_row.get('conflicting_indicators', False)),
        'high_volatility': bool(last_row.get('high_volatility', False)),
        'signal_date': signals.index[-1].date(),
    },
    instrument=instrument_config,
    news=headlines,
)

print(f'\nSignalContext built for {SYMBOL} on {context.signal_date}:')
print(f'  Direction:    {context.signal_direction}')
print(f'  Close:        {context.close:.4f}')
print(f'  RSI:          {context.rsi:.1f}')
print(f'  Confidence:   {context.confidence_score}/100')
print(f'  News items:   {len(context.news_headlines)}')

In [ ]:
# Generate the signal explanation (cache-first)
explanation_service = ExplanationService(client=llm_client, cache_dir=EXPLANATIONS_DIR)
explanation = explanation_service.get_or_generate(context)

print(f"{'='*55}")
print(f"  Signal Explanation — {SYMBOL} ({explanation.signal_date})")
print(f"{'='*55}")
print(f"  Model:   {explanation.model}")
print(f"  Cached:  {explanation.cached}")
print(f"  Signal:  {'+1 (LONG)' if explanation.signal == 1 else '-1 (SHORT)' if explanation.signal == -1 else '0 (FLAT)'}")
print(f"{'─'*55}")
print()
print(explanation.explanation)
print(f"{'='*55}")

---
## Section 11 — LLM Go/No-Go Decision

The decision layer extends the LLM role from explanation to structured judgment. Given the full signal context (indicators + news), it returns one of three outcomes:

| Outcome | Meaning |
|---------|---------|
| `GO` | Evidence supports the technical signal direction |
| `NO_GO` | Evidence contradicts or the risk/reward is unfavourable |
| `UNCERTAIN` | Mixed evidence — the model cannot confidently confirm or deny |

### "No trade" is a first-class outcome

The prompt explicitly instructs the model that `NO_GO` and `UNCERTAIN` are **preferred over a forced GO** when evidence is mixed. This guards against the model rubber-stamping every signal.

### Conflicts with technical signal

If the LLM decision contradicts the SMA crossover direction (e.g. technical says LONG but LLM says NO_GO), `conflicts_with_technical` is set to `True`. This is surfaced as a warning on the dashboard.

### Cache behaviour

Decisions are cached to `data/signals/decisions/<symbol>_<date>.json` — same pattern as explanations.

In [ ]:
# Generate the go/no-go decision (cache-first)
decision_service = DecisionService(client=llm_client, cache_dir=DECISIONS_DIR)
decision = decision_service.get_or_generate(context)

rec_symbol = {'GO': '✅', 'NO_GO': '❌', 'UNCERTAIN': '⚠️'}.get(decision.llm_recommendation, '?')

print(f"{'='*55}")
print(f"  LLM Decision — {SYMBOL} ({decision.signal_date})")
print(f"{'='*55}")
print(f"  Model:                {decision.model}")
print(f"  Cached:               {decision.cached}")
print(f"  Technical signal:     {'+1 (LONG)' if decision.signal == 1 else '-1 (SHORT)' if decision.signal == -1 else '0 (FLAT)'}")
print(f"{'─'*55}")
print(f"  Recommendation:   {rec_symbol}  {decision.llm_recommendation}")
print(f"  Conflicts with technical: {decision.conflicts_with_technical}")
print()
print(f"  Rationale:")
print(f"  {decision.rationale}")
print(f"{'='*55}")

---
## Section 12 — Scratch Space

Use this section for free-form exploration. Some ideas:

- Compare the same strategy across all 5 commodity instruments
- Try different SMA window combinations and compare Sharpe ratios
- Change `rsi_overbought` / `rsi_oversold` thresholds and see the effect on signal count
- Use `ClaudeClient` with a real API key and compare stub vs real explanations
- Plot the RSI alongside signal entries to see the filter in action

**Remember:** Any useful code you write here belongs in `src/trading_lab/`, not in this notebook. Move it once you are happy with it.

In [ ]:
# Your exploration here


---
## Section 13 — Multi-Strategy Signal Comparison

The trading bot is being upgraded from a single SMA crossover strategy to a **multi-strategy ensemble**. Instead of relying on one indicator, the system will compute signals from five independent strategies and pass all of them to Gemini, which acts as the aggregator — issuing GO / NO_GO / UNCERTAIN based on the degree of confluence.

### The five strategies

| # | Strategy | Indicator | Signal fires when... |
|---|----------|-----------|----------------------|
| 1 | **SMA Crossover** | SMA(20/50) | Fast SMA crosses above/below slow SMA |
| 2 | **MACD Crossover** | MACD(12/26/9) | MACD line crosses above/below signal line |
| 3 | **Donchian Channel** | Donchian(20) | Price breaks above 20-day high (long) or below 20-day low (short) |
| 4 | **Bollinger Band Breakout** | Bollinger(20, 2σ) | Price closes above upper band (short — mean reversion) or below lower band (long) |
| 5 | **RSI Mean Reversion** | RSI(14) | RSI crosses back above 30 (long) or back below 70 (short) |

### Signal encoding

All signals are encoded as: `+1` (Long), `-1` (Short), `0` (Flat / no signal).

Note that Bollinger and RSI use **mean-reversion** logic — they fire in the opposite direction to a momentum breakout, betting that price will revert to the mean after an extreme move.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_lab.features.indicators import sma, ema, rsi, macd, atr, bollinger_bands, donchian_channel

# ---------------------------------------------------------------------------
# Load curated data for GC=F (Gold)
# ---------------------------------------------------------------------------
SYMBOL_MS = 'GC=F'
DATA_PATH = ROOT / 'data' / 'curated' / 'gc=f_1d_yfinance.parquet'

df_ms = pd.read_parquet(DATA_PATH)

# Ensure timestamp index
if 'timestamp' in df_ms.columns:
    df_ms = df_ms.set_index('timestamp')
if df_ms.index.tz is None:
    df_ms.index = df_ms.index.tz_localize('UTC')

df_ms = df_ms.sort_index()
print(f'Loaded {len(df_ms)} bars for {SYMBOL_MS}')
print(f'Date range: {df_ms.index.min().date()} → {df_ms.index.max().date()}')
print(f'Columns: {list(df_ms.columns)}')

In [ ]:
# ---------------------------------------------------------------------------
# Compute all five strategy signals
# ---------------------------------------------------------------------------

close = df_ms['close']
high  = df_ms['high']
low   = df_ms['low']

# --- 1. SMA Crossover (20/50) ---
fast_sma_ms = sma(close, 20)
slow_sma_ms = sma(close, 50)
sma_signal = pd.Series(0, index=close.index, dtype=int)
sma_signal[fast_sma_ms > slow_sma_ms] = 1
sma_signal[fast_sma_ms < slow_sma_ms] = -1

# --- 2. MACD Crossover (12/26/9) ---
macd_df = macd(close, fast_window=12, slow_window=26, signal_window=9)
macd_line   = macd_df['macd_line']
signal_line = macd_df['signal_line']
# Signal fires on crossover (not persistent — only on the cross bar)
macd_above_prev = (macd_line.shift(1) <= signal_line.shift(1))
macd_below_prev = (macd_line.shift(1) >= signal_line.shift(1))
macd_signal = pd.Series(0, index=close.index, dtype=int)
macd_signal[(macd_line > signal_line) & macd_above_prev] = 1   # crossed above
macd_signal[(macd_line < signal_line) & macd_below_prev] = -1  # crossed below

# --- 3. Donchian Channel (20) — breakout ---
dc = donchian_channel(high, low, window=20)
# Long when close breaks above the N-day high (on the bar it happens)
# Short when close breaks below the N-day low
# Use previous bar's channel to avoid lookahead
dc_upper_prev = dc['upper'].shift(1)
dc_lower_prev = dc['lower'].shift(1)
donchian_signal = pd.Series(0, index=close.index, dtype=int)
donchian_signal[close > dc_upper_prev] = 1
donchian_signal[close < dc_lower_prev] = -1

# --- 4. Bollinger Band Breakout (20, 2std) — mean reversion ---
bb = bollinger_bands(close, window=20, num_std=2.0)
bb_upper_prev = bb['upper'].shift(1)
bb_lower_prev = bb['lower'].shift(1)
# Mean-reversion: fire LONG when price returns inside from below lower band
# Fire SHORT when price returns inside from above upper band
# i.e. previous bar was outside, current bar is inside
bb_signal = pd.Series(0, index=close.index, dtype=int)
bb_signal[(close.shift(1) < bb_lower_prev.shift(1)) & (close >= bb_lower_prev)] = 1   # re-entered from below
bb_signal[(close.shift(1) > bb_upper_prev.shift(1)) & (close <= bb_upper_prev)] = -1  # re-entered from above

# --- 5. RSI Mean Reversion (14, cross of 30/70) ---
rsi_vals = rsi(close, window=14)
rsi_prev = rsi_vals.shift(1)
rsi_signal = pd.Series(0, index=close.index, dtype=int)
rsi_signal[(rsi_prev < 30) & (rsi_vals >= 30)] = 1   # crossed back above 30 — buy dip
rsi_signal[(rsi_prev > 70) & (rsi_vals <= 70)] = -1  # crossed back below 70 — sell peak

# --- Compile into one DataFrame ---
signals_ms = pd.DataFrame({
    'close':    close,
    'sma':      sma_signal,
    'macd':     macd_signal,
    'donchian': donchian_signal,
    'bollinger':bb_signal,
    'rsi_rev':  rsi_signal,
}, index=close.index)

print('Signal counts per strategy (non-zero bars):')
for col in ['sma', 'macd', 'donchian', 'bollinger', 'rsi_rev']:
    n_long  = (signals_ms[col] ==  1).sum()
    n_short = (signals_ms[col] == -1).sum()
    print(f'  {col:<12}: {n_long:>3} long,  {n_short:>3} short')

In [ ]:
# ---------------------------------------------------------------------------
# Price chart with all 5 signal overlays (Plotly)
# ---------------------------------------------------------------------------

STRATEGY_COLS = ['sma', 'macd', 'donchian', 'bollinger', 'rsi_rev']
STRATEGY_NAMES = {
    'sma':       'SMA 20/50',
    'macd':      'MACD 12/26/9',
    'donchian':  'Donchian(20)',
    'bollinger': 'Bollinger BB',
    'rsi_rev':   'RSI Reversion',
}
MARKER_COLORS = {
    'sma':       ('#2196F3', '#1565C0'),   # blue tones for long/short
    'macd':      ('#FF9800', '#E65100'),   # orange
    'donchian':  ('#4CAF50', '#1B5E20'),   # green
    'bollinger': ('#9C27B0', '#4A148C'),   # purple
    'rsi_rev':   ('#00BCD4', '#006064'),   # cyan
}

fig = go.Figure()

# Candlestick base
fig.add_trace(go.Candlestick(
    x=df_ms.index,
    open=df_ms['open'], high=df_ms['high'],
    low=df_ms['low'],   close=df_ms['close'],
    name='Price', increasing_line_color='#26A69A', decreasing_line_color='#EF5350',
    showlegend=True,
))

offsets = [-1.0, -0.5, 0.0, 0.5, 1.0]  # vertical scatter offset per strategy

for i, col in enumerate(STRATEGY_COLS):
    long_bars  = signals_ms[signals_ms[col] ==  1]
    short_bars = signals_ms[signals_ms[col] == -1]
    color_long, color_short = MARKER_COLORS[col]
    name = STRATEGY_NAMES[col]

    # Offset markers vertically to avoid overlap
    atr_approx = (df_ms['high'] - df_ms['low']).rolling(14).mean()
    offset_val = atr_approx.reindex(long_bars.index).fillna(0) * offsets[i]

    if len(long_bars):
        fig.add_trace(go.Scatter(
            x=long_bars.index,
            y=long_bars['close'] - abs(offset_val.reindex(long_bars.index).fillna(0)) - atr_approx.reindex(long_bars.index).fillna(0),
            mode='markers',
            marker=dict(symbol='triangle-up', size=10, color=color_long),
            name=f'{name} Long',
        ))
    if len(short_bars):
        fig.add_trace(go.Scatter(
            x=short_bars.index,
            y=short_bars['close'] + abs(offset_val.reindex(short_bars.index).fillna(0)) + atr_approx.reindex(short_bars.index).fillna(0),
            mode='markers',
            marker=dict(symbol='triangle-down', size=10, color=color_short),
            name=f'{name} Short',
        ))

fig.update_layout(
    title=f'{SYMBOL_MS} — All 5 Strategy Signals Overlaid on Price',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    height=600,
    legend=dict(orientation='v', x=1.01, y=1),
)
fig.show()

In [ ]:
# ---------------------------------------------------------------------------
# Last 20 bars — all signal columns side by side
# ---------------------------------------------------------------------------

def _fmt_signal(v):
    if v == 1:  return '▲ LONG'
    if v == -1: return '▼ SHORT'
    return '— FLAT'

display_df = signals_ms[['close'] + STRATEGY_COLS].tail(20).copy()
for col in STRATEGY_COLS:
    display_df[col] = display_df[col].apply(_fmt_signal)

display_df.index = display_df.index.date
display_df.columns = ['Close'] + [STRATEGY_NAMES[c] for c in STRATEGY_COLS]
display_df

---
## Section 14 — Strategy Consensus Analysis

When multiple independent strategies agree on a direction simultaneously, the signal is much stronger than any single strategy alone. This section quantifies that agreement using a **consensus score** — the number of strategies pointing in the same direction on each bar.

### How consensus is scored

For each bar, the consensus score is computed as follows:
- Count how many strategies signal **Long** (`+1`)
- Count how many strategies signal **Short** (`-1`)
- Score = `max(long_count, short_count)` — ranging from 0 to 5

A score of 0 means all strategies are flat. A score of 5 means all five strategies agree on the same direction simultaneously — a rare but high-conviction event.

### Why this matters for the LLM aggregator

The Gemini aggregator will receive all five signals and must decide GO / NO_GO / UNCERTAIN. High consensus (4–5 strategies agreeing) makes GO much more likely. Low consensus (1–2 agreeing) makes UNCERTAIN or NO_GO more appropriate.

In [ ]:
# ---------------------------------------------------------------------------
# Compute per-bar consensus score and direction
# ---------------------------------------------------------------------------

sig_matrix = signals_ms[STRATEGY_COLS]

long_count  = (sig_matrix == 1).sum(axis=1)
short_count = (sig_matrix == -1).sum(axis=1)

consensus_score     = np.maximum(long_count, short_count)
consensus_direction = np.where(long_count >= short_count, 1, -1)  # tiebreak to long
consensus_direction = np.where(consensus_score == 0, 0, consensus_direction)

signals_ms['consensus_score']     = consensus_score
signals_ms['consensus_direction'] = consensus_direction

# --- Distribution statistics ---
print('Consensus score distribution:')
print(f"{'Score':<8} {'Bars':>6} {'% of time':>12} {'Direction breakdown'}")
print('─' * 55)
total_bars = len(signals_ms)
for score in range(6):
    mask = signals_ms['consensus_score'] == score
    n = mask.sum()
    pct = n / total_bars * 100
    if score == 0:
        direction_str = 'all flat'
    else:
        n_long  = ((signals_ms['consensus_direction'] == 1)  & mask).sum()
        n_short = ((signals_ms['consensus_direction'] == -1) & mask).sum()
        direction_str = f'{n_long} long, {n_short} short'
    print(f'{score:<8} {n:>6} {pct:>11.1f}%  {direction_str}')

print()
all_agree = signals_ms[signals_ms['consensus_score'] == 5]
print(f'Bars where ALL 5 strategies agree: {len(all_agree)}')
if len(all_agree) > 0:
    print(all_agree[['close', 'consensus_direction'] + STRATEGY_COLS].to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Plot: Price + consensus score over time, highlighting high-agreement bars
# ---------------------------------------------------------------------------

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(f'{SYMBOL_MS} — Close Price with High-Consensus Highlights', 'Consensus Score (0–5)'),
    row_heights=[0.65, 0.35],
    vertical_spacing=0.06,
)

# Price line
fig.add_trace(go.Scatter(
    x=df_ms.index, y=df_ms['close'],
    mode='lines', name='Close', line=dict(color='#607D8B', width=1),
), row=1, col=1)

# Highlight bars where all 5 agree — long
strong_long  = signals_ms[(signals_ms['consensus_score'] == 5) & (signals_ms['consensus_direction'] == 1)]
strong_short = signals_ms[(signals_ms['consensus_score'] == 5) & (signals_ms['consensus_direction'] == -1)]

if len(strong_long):
    fig.add_trace(go.Scatter(
        x=strong_long.index, y=strong_long['close'],
        mode='markers', name='All 5 agree: LONG',
        marker=dict(symbol='star', size=14, color='#4CAF50', line=dict(width=1, color='white')),
    ), row=1, col=1)

if len(strong_short):
    fig.add_trace(go.Scatter(
        x=strong_short.index, y=strong_short['close'],
        mode='markers', name='All 5 agree: SHORT',
        marker=dict(symbol='star', size=14, color='#F44336', line=dict(width=1, color='white')),
    ), row=1, col=1)

# Consensus score bar chart — colour by score level
score_colors = {0: '#ECEFF1', 1: '#B0BEC5', 2: '#78909C', 3: '#FF9800', 4: '#F44336', 5: '#9C27B0'}
for score_val in range(6):
    mask = signals_ms['consensus_score'] == score_val
    subset = signals_ms[mask]
    if len(subset):
        fig.add_trace(go.Bar(
            x=subset.index,
            y=[score_val] * len(subset),
            name=f'Score {score_val}',
            marker_color=score_colors[score_val],
            showlegend=(score_val >= 3),
        ), row=2, col=1)

fig.add_hline(y=4, line_dash='dot', line_color='orange', row=2, col=1,
              annotation_text='Strong signal threshold (4+)', annotation_position='top right')

fig.update_layout(
    barmode='overlay',
    height=650,
    title=f'{SYMBOL_MS} — Consensus Score Over Time',
    legend=dict(orientation='v', x=1.01, y=1),
)
fig.update_yaxes(title_text='Price', row=1, col=1)
fig.update_yaxes(title_text='Agreement (0–5)', range=[0, 5.5], row=2, col=1)
fig.show()

---
## Section 15 — Backtest Comparison — All Strategies

This section runs a simplified bar-by-bar backtest for each strategy independently and compares their performance on the same Gold (GC=F) dataset.

### Backtest methodology

The backtest is intentionally simple here — it does not use the full `BacktestConfig` / `run_backtest` machinery (which is designed for the persistent SMA signal format). Instead, it simulates a **signal-on-cross** approach:

- When a non-zero signal fires, the position changes on the **next bar open** (one-bar lag, no lookahead)
- A round-trip transaction cost of **7 bps** is deducted on every position change
- All strategies start with £100,000 and run over the same full dataset

### What to compare

| Metric | What it tells you |
|--------|------------------|
| **Total return** | Raw profit/loss for the period |
| **Sharpe ratio** | Risk-adjusted performance — higher is better |
| **Max drawdown** | Worst peak-to-trough loss — lower is better |
| **Trade count** | Frequency of signals — more trades = more costs |

Note that the SMA signal here is *persistent* (position stays long/short until a new crossover), while MACD, Donchian, Bollinger, and RSI signals are *one-bar* events that revert to flat the next bar. This is reflected in the trade counts.

In [ ]:
# ---------------------------------------------------------------------------
# Simplified backtest function for single-strategy evaluation
# ---------------------------------------------------------------------------

def simple_backtest(signal_series: pd.Series, price_series: pd.Series,
                    initial_cash: float = 100_000,
                    cost_bps: float = 7) -> dict:
    """
    Returns-based bar-by-bar backtest.
    - signal_series: +1 (long), -1 (short), 0 (flat) per bar
    - price_series:  close prices aligned to signal_series
    - cost_bps:      round-trip transaction cost in basis points (applied on change)
    Returns a dict of performance metrics and an equity Series.
    """
    sig   = signal_series.reindex(price_series.index).fillna(0).astype(int)
    price = price_series.copy()

    # Daily price returns
    price_ret = price.pct_change().fillna(0)

    # Position on bar i is the signal from bar i-1 (one-bar execution lag)
    position = sig.shift(1).fillna(0)

    # Strategy return per bar = position * market return
    strat_ret = position * price_ret

    # Deduct transaction costs on every position change (one-way cost_bps per side)
    pos_change = position.diff().abs().fillna(0)
    strat_ret -= pos_change * (cost_bps / 10_000)

    # Equity curve from compounded returns
    equity_s = initial_cash * (1 + strat_ret).cumprod()
    equity_s.iloc[0] = initial_cash  # anchor first bar

    daily_ret  = equity_s.pct_change().dropna()
    total_ret  = (equity_s.iloc[-1] / initial_cash - 1) * 100
    n_years    = (price.index[-1] - price.index[0]).days / 365.25
    cagr       = ((equity_s.iloc[-1] / initial_cash) ** (1 / max(n_years, 0.01)) - 1) * 100
    sharpe     = (daily_ret.mean() / daily_ret.std() * np.sqrt(252)) if daily_ret.std() > 0 else 0.0
    peak       = equity_s.cummax()
    max_dd     = ((equity_s - peak) / peak * 100).min()
    trade_count = int((pos_change > 0).sum())

    return {
        'equity':       equity_s,
        'total_return': total_ret,
        'cagr':         cagr,
        'sharpe':       sharpe,
        'max_drawdown': max_dd,
        'trades':       trade_count,
    }


# ---------------------------------------------------------------------------
# Run backtest for each strategy
# ---------------------------------------------------------------------------

INITIAL_CASH = 100_000

# SMA is persistent — carry forward the last non-zero signal
sma_persistent = sma_signal.replace(0, np.nan).ffill().fillna(0).astype(int)

bt_signals = {
    'SMA 20/50':     sma_persistent,
    'MACD 12/26/9':  macd_signal,
    'Donchian(20)':  donchian_signal,
    'Bollinger BB':  bb_signal,
    'RSI Reversion': rsi_signal,
}

results_bt = {}
for name, sig in bt_signals.items():
    res = simple_backtest(sig, close, initial_cash=INITIAL_CASH)
    results_bt[name] = res

# --- Comparison table ---
print(f"{'Strategy':<18} {'Total Ret%':>11} {'CAGR%':>8} {'Sharpe':>8} {'Max DD%':>9} {'Trades':>8}")
print('─' * 68)
for name, res in results_bt.items():
    print(f"{name:<18} {res['total_return']:>10.1f}% {res['cagr']:>7.1f}% "
          f"{res['sharpe']:>8.2f} {res['max_drawdown']:>8.1f}% {res['trades']:>8}")


In [ ]:
# ---------------------------------------------------------------------------
# Equity curves — all 5 strategies on one chart
# ---------------------------------------------------------------------------

EQUITY_COLORS = {
    'SMA 20/50':     '#2196F3',
    'MACD 12/26/9':  '#FF9800',
    'Donchian(20)':  '#4CAF50',
    'Bollinger BB':  '#9C27B0',
    'RSI Reversion': '#00BCD4',
}

fig = go.Figure()

fig.add_hline(y=INITIAL_CASH, line_dash='dot', line_color='grey',
              annotation_text='Starting capital', annotation_position='top left')

for name, res in results_bt.items():
    fig.add_trace(go.Scatter(
        x=res['equity'].index,
        y=res['equity'].values,
        mode='lines',
        name=name,
        line=dict(color=EQUITY_COLORS.get(name, '#333'), width=1.5),
    ))

fig.update_layout(
    title=f'{SYMBOL_MS} — Equity Curves: All 5 Strategies (£{INITIAL_CASH:,} start)',
    xaxis_title='Date',
    yaxis_title='Portfolio Value (£)',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## Section 16 — Indicator Visualisation

This section provides detailed visualisations of each new indicator: what the raw values look like, and how the signal threshold or crossover appears on the chart. Understanding the indicator visually is essential before trusting its signals.

### MACD (Moving Average Convergence Divergence)

**What it is:** MACD measures momentum by computing the difference between a 12-day EMA and 26-day EMA. A 9-day EMA of the MACD line (the "signal line") acts as a trigger.

**How it fires:** When the MACD line crosses above the signal line, momentum is turning bullish. When it crosses below, momentum is turning bearish. The histogram (MACD - signal) makes crossovers easy to spot — it crosses zero at the same moment.

**Strengths:** Adapts to recent price behaviour via the EMA. Good at capturing momentum shifts.

**Weaknesses:** Lags price — like all moving average systems, it generates signals after the move has started. Prone to whipsaws in sideways markets.

---

### Bollinger Bands (BB)

**What it is:** A 20-day SMA (the middle band) surrounded by upper and lower bands placed 2 standard deviations away. The bands widen during volatile periods and narrow during quiet periods.

**How it fires (mean reversion):** In this system, Bollinger Bands are used as a **mean reversion** indicator. The signal fires when price *re-enters* the bands after a brief excursion outside them — betting that the extreme move is exhausted.

**Strengths:** Statistically grounded — price spends approximately 95% of the time inside 2-std bands by construction. Adapts to volatility.

**Weaknesses:** In a strong trend, price can "walk the band" — stay outside for extended periods. Mean reversion bets against strong trends.

---

### Donchian Channel

**What it is:** The 20-day highest high and 20-day lowest low, forming a channel. The midpoint is the average of the two.

**How it fires (breakout):** A long signal fires when price breaks above the 20-day high. A short signal fires when price breaks below the 20-day low. This is classic **turtle trader** logic.

**Strengths:** Simple, objective, no parameters to overfit. Works extremely well in strongly trending markets.

**Weaknesses:** In sideways markets, every small swing high/low generates a false breakout. High trade frequency = high costs.

---

### RSI (Mean Reversion)

Already shown in Section 4, but here we focus on the **signal crossback** logic rather than the raw RSI level. Long fires when RSI crosses back above 30 (oversold exhaustion). Short fires when RSI crosses back below 70 (overbought exhaustion).

In [ ]:
# ---------------------------------------------------------------------------
# Plot 1: MACD — line, signal line, and histogram
# ---------------------------------------------------------------------------

fig_macd = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=(f'{SYMBOL_MS} — Close Price', 'MACD (12/26/9)'),
    row_heights=[0.5, 0.5],
    vertical_spacing=0.06,
)

# Price
fig_macd.add_trace(go.Scatter(
    x=close.index, y=close.values,
    mode='lines', name='Close', line=dict(color='#607D8B', width=1),
), row=1, col=1)

# MACD line and signal line
fig_macd.add_trace(go.Scatter(
    x=macd_line.index, y=macd_line.values,
    mode='lines', name='MACD line', line=dict(color='#2196F3', width=1.5),
), row=2, col=1)

fig_macd.add_trace(go.Scatter(
    x=signal_line.index, y=signal_line.values,
    mode='lines', name='Signal line', line=dict(color='#FF9800', width=1.5),
), row=2, col=1)

# Histogram — colour positive/negative differently
histogram = macd_df['histogram']
pos_hist = histogram.clip(lower=0)
neg_hist = histogram.clip(upper=0)

fig_macd.add_trace(go.Bar(
    x=histogram.index, y=pos_hist.values,
    name='Histogram (+)', marker_color='rgba(76,175,80,0.6)',
), row=2, col=1)
fig_macd.add_trace(go.Bar(
    x=histogram.index, y=neg_hist.values,
    name='Histogram (−)', marker_color='rgba(244,67,54,0.6)',
), row=2, col=1)

fig_macd.add_hline(y=0, line_dash='dot', line_color='grey', row=2, col=1)

# Mark crossover bars
macd_long_bars  = signals_ms[signals_ms['macd'] ==  1]
macd_short_bars = signals_ms[signals_ms['macd'] == -1]

if len(macd_long_bars):
    fig_macd.add_trace(go.Scatter(
        x=macd_long_bars.index, y=macd_long_bars['close'],
        mode='markers', name='MACD Long cross',
        marker=dict(symbol='triangle-up', size=10, color='#4CAF50'),
    ), row=1, col=1)

if len(macd_short_bars):
    fig_macd.add_trace(go.Scatter(
        x=macd_short_bars.index, y=macd_short_bars['close'],
        mode='markers', name='MACD Short cross',
        marker=dict(symbol='triangle-down', size=10, color='#F44336'),
    ), row=1, col=1)

fig_macd.update_layout(
    height=600, barmode='relative',
    title=f'{SYMBOL_MS} — MACD Indicator (12/26/9)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig_macd.update_yaxes(title_text='Price', row=1, col=1)
fig_macd.update_yaxes(title_text='MACD', row=2, col=1)
fig_macd.show()

In [ ]:
# ---------------------------------------------------------------------------
# Plot 2: Bollinger Bands — upper, middle, lower with price
# ---------------------------------------------------------------------------

fig_bb = go.Figure()

# Price
fig_bb.add_trace(go.Scatter(
    x=close.index, y=close.values,
    mode='lines', name='Close',
    line=dict(color='#607D8B', width=1.5),
))

# Upper band
fig_bb.add_trace(go.Scatter(
    x=bb['upper'].index, y=bb['upper'].values,
    mode='lines', name='Upper band (2σ)',
    line=dict(color='#F44336', width=1, dash='dash'),
))

# Lower band with fill to upper
fig_bb.add_trace(go.Scatter(
    x=bb['lower'].index, y=bb['lower'].values,
    mode='lines', name='Lower band (2σ)',
    line=dict(color='#4CAF50', width=1, dash='dash'),
    fill='tonexty',
    fillcolor='rgba(33,150,243,0.07)',
))

# Middle band (SMA)
fig_bb.add_trace(go.Scatter(
    x=bb['middle'].index, y=bb['middle'].values,
    mode='lines', name='Middle (SMA 20)',
    line=dict(color='#2196F3', width=1),
))

# Mark signal bars (re-entry)
bb_long_bars  = signals_ms[signals_ms['bollinger'] ==  1]
bb_short_bars = signals_ms[signals_ms['bollinger'] == -1]

if len(bb_long_bars):
    fig_bb.add_trace(go.Scatter(
        x=bb_long_bars.index, y=bb_long_bars['close'],
        mode='markers', name='BB Mean Rev: Long',
        marker=dict(symbol='triangle-up', size=12, color='#4CAF50',
                    line=dict(width=1, color='white')),
    ))
if len(bb_short_bars):
    fig_bb.add_trace(go.Scatter(
        x=bb_short_bars.index, y=bb_short_bars['close'],
        mode='markers', name='BB Mean Rev: Short',
        marker=dict(symbol='triangle-down', size=12, color='#F44336',
                    line=dict(width=1, color='white')),
    ))

fig_bb.update_layout(
    title=f'{SYMBOL_MS} — Bollinger Bands (20 period, 2 std dev)',
    xaxis_title='Date',
    yaxis_title='Price',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig_bb.show()

# --- Band touch statistics ---
outside_upper = (close > bb['upper']).sum()
outside_lower = (close < bb['lower']).sum()
total_valid   = close[bb['upper'].notna()].count()
print(f'Days above upper band: {outside_upper} ({outside_upper/total_valid*100:.1f}%)')
print(f'Days below lower band: {outside_lower} ({outside_lower/total_valid*100:.1f}%)')
print(f'Days inside bands:     {total_valid - outside_upper - outside_lower} ({(total_valid - outside_upper - outside_lower)/total_valid*100:.1f}%)')

In [ ]:
# ---------------------------------------------------------------------------
# Plot 3: Donchian Channel — upper, lower, midline with price
# ---------------------------------------------------------------------------

fig_dc = go.Figure()

# Price (candlestick)
fig_dc.add_trace(go.Candlestick(
    x=df_ms.index,
    open=df_ms['open'], high=df_ms['high'],
    low=df_ms['low'],   close=df_ms['close'],
    name='Price',
    increasing_line_color='#26A69A', decreasing_line_color='#EF5350',
    showlegend=True,
))

# Donchian upper channel
fig_dc.add_trace(go.Scatter(
    x=dc['upper'].index, y=dc['upper'].values,
    mode='lines', name='20-day High (channel top)',
    line=dict(color='#F44336', width=1.5),
))

# Donchian lower — fill to upper
fig_dc.add_trace(go.Scatter(
    x=dc['lower'].index, y=dc['lower'].values,
    mode='lines', name='20-day Low (channel bottom)',
    line=dict(color='#4CAF50', width=1.5),
    fill='tonexty',
    fillcolor='rgba(255,235,59,0.08)',
))

# Midline
fig_dc.add_trace(go.Scatter(
    x=dc['middle'].index, y=dc['middle'].values,
    mode='lines', name='Channel midline',
    line=dict(color='#FF9800', width=1, dash='dot'),
))

# Breakout signal markers
dc_long_bars  = signals_ms[signals_ms['donchian'] ==  1]
dc_short_bars = signals_ms[signals_ms['donchian'] == -1]

if len(dc_long_bars):
    fig_dc.add_trace(go.Scatter(
        x=dc_long_bars.index, y=dc_long_bars['close'],
        mode='markers', name='Donchian Long breakout',
        marker=dict(symbol='triangle-up', size=12, color='#4CAF50',
                    line=dict(width=1, color='white')),
    ))
if len(dc_short_bars):
    fig_dc.add_trace(go.Scatter(
        x=dc_short_bars.index, y=dc_short_bars['close'],
        mode='markers', name='Donchian Short breakout',
        marker=dict(symbol='triangle-down', size=12, color='#F44336',
                    line=dict(width=1, color='white')),
    ))

fig_dc.update_layout(
    title=f'{SYMBOL_MS} — Donchian Channel (20-day)',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig_dc.show()

print(f'Donchian breakout signals: {(donchian_signal == 1).sum()} long, {(donchian_signal == -1).sum()} short')

In [ ]:
# ---------------------------------------------------------------------------
# Plot 4: RSI with overbought/oversold lines and mean-reversion signal markers
# ---------------------------------------------------------------------------

fig_rsi = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=(f'{SYMBOL_MS} — Close Price', 'RSI (14) — Mean Reversion Signal'),
    row_heights=[0.5, 0.5],
    vertical_spacing=0.06,
)

# Price
fig_rsi.add_trace(go.Scatter(
    x=close.index, y=close.values,
    mode='lines', name='Close', line=dict(color='#607D8B', width=1),
), row=1, col=1)

# Mark RSI signal bars on price
rsi_long_bars  = signals_ms[signals_ms['rsi_rev'] ==  1]
rsi_short_bars = signals_ms[signals_ms['rsi_rev'] == -1]

if len(rsi_long_bars):
    fig_rsi.add_trace(go.Scatter(
        x=rsi_long_bars.index, y=rsi_long_bars['close'],
        mode='markers', name='RSI crossed above 30 (Long)',
        marker=dict(symbol='triangle-up', size=12, color='#4CAF50',
                    line=dict(width=1, color='white')),
    ), row=1, col=1)

if len(rsi_short_bars):
    fig_rsi.add_trace(go.Scatter(
        x=rsi_short_bars.index, y=rsi_short_bars['close'],
        mode='markers', name='RSI crossed below 70 (Short)',
        marker=dict(symbol='triangle-down', size=12, color='#F44336',
                    line=dict(width=1, color='white')),
    ), row=1, col=1)

# RSI line
fig_rsi.add_trace(go.Scatter(
    x=rsi_vals.index, y=rsi_vals.values,
    mode='lines', name='RSI (14)', line=dict(color='#9C27B0', width=1.5),
), row=2, col=1)

# Threshold lines
fig_rsi.add_hline(y=70, line_dash='dash', line_color='#F44336', row=2, col=1,
                  annotation_text='Overbought (70)', annotation_position='top right')
fig_rsi.add_hline(y=30, line_dash='dash', line_color='#4CAF50', row=2, col=1,
                  annotation_text='Oversold (30)', annotation_position='bottom right')
fig_rsi.add_hline(y=50, line_dash='dot', line_color='grey', row=2, col=1)

# Shaded regions
fig_rsi.add_hrect(y0=70, y1=100, line_width=0, fillcolor='red', opacity=0.05, row=2, col=1)
fig_rsi.add_hrect(y0=0,  y1=30,  line_width=0, fillcolor='green', opacity=0.05, row=2, col=1)

# Mark signal crossback points on RSI panel
if len(rsi_long_bars):
    fig_rsi.add_trace(go.Scatter(
        x=rsi_long_bars.index, y=rsi_vals.reindex(rsi_long_bars.index),
        mode='markers', name='Cross above 30',
        marker=dict(symbol='circle', size=8, color='#4CAF50'),
        showlegend=False,
    ), row=2, col=1)

if len(rsi_short_bars):
    fig_rsi.add_trace(go.Scatter(
        x=rsi_short_bars.index, y=rsi_vals.reindex(rsi_short_bars.index),
        mode='markers', name='Cross below 70',
        marker=dict(symbol='circle', size=8, color='#F44336'),
        showlegend=False,
    ), row=2, col=1)

fig_rsi.update_layout(
    title=f'{SYMBOL_MS} — RSI (14) Mean Reversion Signal',
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig_rsi.update_yaxes(title_text='Price', row=1, col=1)
fig_rsi.update_yaxes(title_text='RSI', range=[0, 100], row=2, col=1)
fig_rsi.show()

# Stats
time_overbought = (rsi_vals > 70).sum()
time_oversold   = (rsi_vals < 30).sum()
total_rsi       = rsi_vals.notna().sum()
print(f'RSI time above 70 (overbought): {time_overbought} bars ({time_overbought/total_rsi*100:.1f}%)')
print(f'RSI time below 30 (oversold):   {time_oversold} bars ({time_oversold/total_rsi*100:.1f}%)')
print(f'RSI mean-reversion signals:     {(rsi_signal == 1).sum()} long, {(rsi_signal == -1).sum()} short')
print(f'Current RSI:                    {rsi_vals.iloc[-1]:.1f}')